# LeetCode 4106 — Find Mismatched Segments Between Two Strings (Medium)

Bài toán LeetCode:

```python
class Solution:
    def findMismatchedSegments(self, s1: str, s2: str) -> List[List[str]]:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4106 (Weekly Contest 527 — câu 1).

Bài này luyện một kỹ năng xuất hiện ở rất nhiều bài chuỗi/mảng: **tách dãy thành các "đoạn liên tiếp tối đa" (maximal runs)** thỏa một điều kiện. Ta sẽ đi từ cách làm "dịch nguyên văn định nghĩa" sang các cách quét một lượt gọn gàng: biến trạng thái, hai con trỏ, và `itertools.groupby`.

---

## Sau notebook này, bạn sẽ nắm được

1. "Đoạn liên tiếp tối đa" nghĩa là gì và cách kiểm tra một đoạn có tối đa hay không.
2. Các công cụ Python: chỉ số chuỗi, `zip`, `enumerate`, slicing `s[i:j]`, `itertools.groupby`.
3. Cách 1 — Brute Force dịch thẳng định nghĩa (`O(n³)`).
4. Cách 2 — Quét một lượt với biến `start` (máy trạng thái "trong đoạn / ngoài đoạn") và mẹo lính canh.
5. Cách 3 — Hai con trỏ `i`, `j`.
6. Cách 4 — `itertools.groupby` gom nhóm tự động.
7. Lỗi hay gặp (quên đoạn cuối, lệch chỉ số slicing...), mẹo nhận diện và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai chuỗi `s1` và `s2` **cùng độ dài** `n`.

Một **đoạn lệch (mismatched segment)** là một dãy chỉ số **liên tiếp** `[i, j]` sao cho:

- tại **mọi** vị trí `k` trong đoạn, `s1[k] != s2[k]`;
- đoạn là **tối đa (maximal)**: không thể kéo dài thêm sang trái hay sang phải mà vẫn thỏa điều kiện trên.

Với mỗi đoạn lệch, trả về cặp `[s1[i..j], s2[i..j]]` — phần chuỗi tương ứng của `s1` và `s2`. Các cặp được sắp theo vị trí bắt đầu tăng dần. Nếu hai chuỗi giống hệt nhau, trả về `[]`.

---

## Ví dụ 1

```python
s1 = "earth", s2 = "empty"   ->   [["ar", "mp"], ["h", "y"]]
```

| chỉ số | 0 | 1 | 2 | 3 | 4 |
|---|:---:|:---:|:---:|:---:|:---:|
| `s1` | e | a | r | t | h |
| `s2` | e | m | p | t | y |
| lệch? | | ✗ | ✗ | | ✗ |

Các vị trí lệch là `1, 2` (liền nhau → một đoạn) và `4` (một đoạn riêng).

## Ví dụ 2

```python
s1 = "action", s2 = "nation"   ->   [["ac", "na"]]
```

Chỉ hai vị trí đầu lệch, phần `"tion"` giống nhau.

## Ví dụ 3

```python
s1 = "hot", s2 = "hot"   ->   []
```

---

## Ràng buộc

```text
1 <= s1.length == s2.length <= 100
s1, s2 chỉ gồm chữ cái tiếng Anh viết thường
```

`n <= 100` rất nhỏ, nên **cách nào đúng cũng qua**. Mục tiêu thật sự của bài là viết code **đúng, gọn, không sót trường hợp biên** — kỹ năng sẽ dùng lại ở các bài lớn hơn.

# 2. Kiến thức Python nền tảng

## 2.1. So sánh từng vị trí của hai chuỗi

`s1[k]` là ký tự thứ `k` (đếm từ `0`). Hai ký tự so sánh bằng `==` / `!=`.

In [1]:
s1, s2 = "earth", "empty"

for k in range(len(s1)):
    mark = "LỆCH" if s1[k] != s2[k] else "khớp"
    print(f"k = {k}: {s1[k]} vs {s2[k]} -> {mark}")

k = 0: e vs e -> khớp
k = 1: a vs m -> LỆCH
k = 2: r vs p -> LỆCH
k = 3: t vs t -> khớp
k = 4: h vs y -> LỆCH


## 2.2. `zip` và `enumerate`

- `zip(s1, s2)` đi song song hai chuỗi, mỗi bước cho một cặp `(c1, c2)`.
- `enumerate(...)` thêm chỉ số đi kèm.

Kết hợp lại: `for k, (c1, c2) in enumerate(zip(s1, s2))`.

In [2]:
for k, (c1, c2) in enumerate(zip("action", "nation")):
    print(k, c1, c2, c1 != c2)

# Một dòng: danh sách cờ lệch
flags = [c1 != c2 for c1, c2 in zip("earth", "empty")]
print(flags)

0 a n True
1 c a True
2 t t False
3 i i False
4 o o False
5 n n False
[False, True, True, False, True]


## 2.3. Slicing `s[i:j]` lấy từ `i` tới **trước** `j`

Đoạn chỉ số `[i, j]` (tính cả `j`) tương ứng với `s[i:j + 1]`. Đây là chỗ cực dễ lệch một đơn vị.

In [3]:
s = "earth"
print(s[1:3])     # 'ar'  -> chỉ số 1, 2
print(s[4:5])     # 'h'   -> chỉ số 4
print(s[1:1])     # ''    -> đoạn rỗng

ar
h



## 2.4. `itertools.groupby` — gom các phần tử liền nhau có cùng "nhãn"

`groupby(iterable, key=f)` duyệt từ trái sang phải và gom các phần tử **liền nhau** có cùng giá trị `f(x)` thành một nhóm. Mỗi bước trả về `(nhãn, nhóm)`, với `nhóm` là một iterator (dùng `list(...)` để xem).

Lưu ý: `groupby` chỉ gom phần tử **liền kề**, không gom các phần tử cùng nhãn nằm rải rác.

In [4]:
from itertools import groupby

for label, grp in groupby("aaabccaa"):
    print(label, list(grp))

print("---")
for is_even, grp in groupby([2, 4, 1, 3, 5, 6], key=lambda x: x % 2 == 0):
    print(is_even, list(grp))

a ['a', 'a', 'a']
b ['b']
c ['c', 'c']
a ['a', 'a']
---
True [2, 4]
False [1, 3, 5]
True [6]


# 3. Cách 1 — Brute Force: dịch nguyên văn định nghĩa

Định nghĩa nói: đoạn `[i, j]` là đoạn lệch nếu

1. mọi vị trí trong `[i, j]` đều lệch, **và**
2. không kéo dài được: vị trí `i - 1` (nếu có) khớp, và vị trí `j + 1` (nếu có) khớp.

Cách "thật thà" nhất là thử **mọi** cặp `(i, j)` và kiểm tra đúng hai điều kiện này. Cách này chậm nhưng rất có giá trị: nó là **chuẩn đối chiếu** để kiểm tra các cách nhanh hơn.

In [5]:
from typing import List


def mismatched_bruteforce(s1: str, s2: str) -> List[List[str]]:
    n = len(s1)
    result = []
    for i in range(n):
        for j in range(i, n):
            all_diff = all(s1[k] != s2[k] for k in range(i, j + 1))
            left_ok = (i == 0) or (s1[i - 1] == s2[i - 1])       # không kéo sang trái được
            right_ok = (j == n - 1) or (s1[j + 1] == s2[j + 1])  # không kéo sang phải được
            if all_diff and left_ok and right_ok:
                result.append([s1[i:j + 1], s2[i:j + 1]])
    return result


print(mismatched_bruteforce("earth", "empty"))
print(mismatched_bruteforce("action", "nation"))
print(mismatched_bruteforce("hot", "hot"))

[['ar', 'mp'], ['h', 'y']]
[['ac', 'na']]
[]


## Độ phức tạp

Có `O(n²)` cặp `(i, j)`, mỗi cặp kiểm tra `all(...)` tốn `O(n)` → **`O(n³)`**. Với `n = 100` là khoảng một triệu bước — vẫn qua, nhưng nếu đề nâng lên `n = 10^5` thì hết đường.

Điểm lãng phí: mỗi vị trí bị kiểm tra "lệch hay không" **rất nhiều lần**. Thực ra chỉ cần nhìn mỗi vị trí **một lần** từ trái sang phải.

---

# 4. Bước đột phá: quét một lượt, nhớ "đang ở trong đoạn hay không"

Đi từ trái sang phải trên dãy cờ lệch:

```text
chỉ số : 0  1  2  3  4
lệch?  : .  X  X  .  X
```

Tại mỗi vị trí chỉ có 4 tình huống, tùy theo **trạng thái hiện tại** (đang trong đoạn lệch hay không) và **vị trí này lệch hay khớp**:

| Đang trong đoạn? | Vị trí này | Hành động |
|:---:|:---:|---|
| không | khớp | không làm gì |
| không | **lệch** | **mở đoạn**: `start = k` |
| có | lệch | không làm gì (đoạn kéo dài) |
| có | **khớp** | **đóng đoạn** `[start, k - 1]` |

Đây là một **máy trạng thái (state machine)** hai trạng thái. Ta dùng `start = -1` để biểu diễn "không trong đoạn nào".

Còn một chi tiết: nếu chuỗi **kết thúc khi đoạn vẫn đang mở** (ví dụ 1: vị trí cuối `4` lệch), ta phải đóng đoạn sau vòng lặp.

# 5. Cách 2 — Biến `start` + đóng đoạn cuối

In [6]:
def mismatched_state(s1: str, s2: str) -> List[List[str]]:
    n = len(s1)
    result = []
    start = -1                                   # -1: không trong đoạn lệch nào
    for k in range(n):
        if s1[k] != s2[k]:
            if start == -1:
                start = k                        # mở đoạn
        else:
            if start != -1:
                result.append([s1[start:k], s2[start:k]])   # đóng đoạn [start, k-1]
                start = -1
    if start != -1:                              # đoạn còn mở tới cuối chuỗi
        result.append([s1[start:], s2[start:]])
    return result


print(mismatched_state("earth", "empty"))
print(mismatched_state("action", "nation"))
print(mismatched_state("hot", "hot"))

[['ar', 'mp'], ['h', 'y']]
[['ac', 'na']]
[]


Để ý cách slicing: khi đóng đoạn tại vị trí khớp `k`, đoạn lệch là `[start, k - 1]`, tương ứng `s1[start:k]` — **không** cần `+ 1` vì `k` đã là vị trí đầu tiên **ngoài** đoạn.

## Mô phỏng với ví dụ 1

`s1 = "earth"`, `s2 = "empty"`

| k | s1[k] | s2[k] | lệch? | start trước | hành động | result |
|---:|:---:|:---:|:---:|---:|---|---|
| 0 | e | e | | -1 | — | [] |
| 1 | a | m | ✗ | -1 | mở, start = 1 | [] |
| 2 | r | p | ✗ | 1 | kéo dài | [] |
| 3 | t | t | | 1 | đóng `[1, 2]` → `["ar", "mp"]`, start = -1 | [["ar","mp"]] |
| 4 | h | y | ✗ | -1 | mở, start = 4 | [["ar","mp"]] |
| hết | | | | 4 | đóng đoạn cuối `[4, 4]` → `["h", "y"]` | [["ar","mp"], ["h","y"]] |

In [7]:
def trace_state(s1: str, s2: str) -> List[List[str]]:
    n = len(s1)
    result = []
    start = -1
    for k in range(n):
        diff = s1[k] != s2[k]
        before = start
        if diff and start == -1:
            start = k
            action = f"mở đoạn, start = {k}"
        elif diff:
            action = "kéo dài đoạn"
        elif start != -1:
            result.append([s1[start:k], s2[start:k]])
            action = f"đóng [{start}, {k - 1}] -> {result[-1]}"
            start = -1
        else:
            action = "—"
        print(f"k = {k}: {s1[k]} vs {s2[k]} | {'lệch' if diff else 'khớp'} | start {before:>2} -> {start:>2} | {action}")
    if start != -1:
        result.append([s1[start:], s2[start:]])
        print(f"hết chuỗi: đóng đoạn cuối [{start}, {n - 1}] -> {result[-1]}")
    print("Kết quả:", result)
    return result


trace_state("earth", "empty");

k = 0: e vs e | khớp | start -1 -> -1 | —
k = 1: a vs m | lệch | start -1 ->  1 | mở đoạn, start = 1
k = 2: r vs p | lệch | start  1 ->  1 | kéo dài đoạn
k = 3: t vs t | khớp | start  1 -> -1 | đóng [1, 2] -> ['ar', 'mp']
k = 4: h vs y | lệch | start -1 ->  4 | mở đoạn, start = 4
hết chuỗi: đóng đoạn cuối [4, 4] -> ['h', 'y']
Kết quả: [['ar', 'mp'], ['h', 'y']]


## Mẹo lính canh: không cần đóng đoạn cuối riêng

Đoạn code "đóng đoạn sau vòng lặp" rất hay bị quên. Mẹo: **thêm một vị trí giả ở cuối mà chắc chắn khớp** (ví dụ thêm cùng ký tự `'#'` vào cuối cả hai chuỗi). Khi đó mọi đoạn đều được đóng bên trong vòng lặp.

Trong code, thay vì nối chuỗi, ta có thể cho `k` chạy tới `n` và coi vị trí `n` là "khớp":

In [8]:
def mismatched_sentinel(s1: str, s2: str) -> List[List[str]]:
    n = len(s1)
    result = []
    start = -1
    for k in range(n + 1):                       # k = n là vị trí lính canh
        diff = k < n and s1[k] != s2[k]          # lính canh luôn được coi là "khớp"
        if diff and start == -1:
            start = k
        elif not diff and start != -1:
            result.append([s1[start:k], s2[start:k]])
            start = -1
    return result


print(mismatched_sentinel("earth", "empty"))
print(mismatched_sentinel("action", "nation"))
print(mismatched_sentinel("hot", "hot"))

[['ar', 'mp'], ['h', 'y']]
[['ac', 'na']]
[]


# 6. Cách 3 — Hai con trỏ `i`, `j`

Một cách viết khác rất phổ biến cho bài "tách đoạn tối đa":

1. Con trỏ `i` đi tìm **đầu đoạn**: nếu `s1[i] == s2[i]` thì `i += 1`.
2. Gặp vị trí lệch tại `i` → cho con trỏ `j` chạy từ `i` sang phải **chừng nào còn lệch**.
3. Khi `j` dừng (gặp vị trí khớp hoặc hết chuỗi), đoạn lệch là `[i, j - 1]` → ghi `s1[i:j]`, `s2[i:j]`.
4. Nhảy `i = j` và lặp lại.

Ưu điểm: không cần biến trạng thái, không cần đóng đoạn cuối riêng — điều kiện `j < n` trong vòng lặp trong tự xử lý.

Vì `i` luôn nhảy tới `j`, **mỗi vị trí chỉ được nhìn một lần** → `O(n)`, dù có hai vòng `while` lồng nhau.

In [9]:
def mismatched_two_pointers(s1: str, s2: str) -> List[List[str]]:
    n = len(s1)
    result = []
    i = 0
    while i < n:
        if s1[i] == s2[i]:
            i += 1                               # chưa phải đầu đoạn
            continue
        j = i
        while j < n and s1[j] != s2[j]:          # kéo hết đoạn lệch
            j += 1
        result.append([s1[i:j], s2[i:j]])        # đoạn [i, j-1]
        i = j                                    # nhảy qua cả đoạn
    return result


print(mismatched_two_pointers("earth", "empty"))
print(mismatched_two_pointers("action", "nation"))
print(mismatched_two_pointers("hot", "hot"))

[['ar', 'mp'], ['h', 'y']]
[['ac', 'na']]
[]


In [10]:
def trace_two_pointers(s1: str, s2: str) -> List[List[str]]:
    n = len(s1)
    result = []
    i = 0
    while i < n:
        if s1[i] == s2[i]:
            print(f"i = {i}: {s1[i]} == {s2[i]} -> khớp, i += 1")
            i += 1
            continue
        j = i
        while j < n and s1[j] != s2[j]:
            j += 1
        result.append([s1[i:j], s2[i:j]])
        print(f"i = {i}: lệch -> j chạy tới {j} -> đoạn [{i}, {j - 1}] = {result[-1]}, nhảy i = {j}")
        i = j
    print("Kết quả:", result)
    return result


trace_two_pointers("abcdef", "xbzzez");

i = 0: lệch -> j chạy tới 1 -> đoạn [0, 0] = ['a', 'x'], nhảy i = 1
i = 1: b == b -> khớp, i += 1
i = 2: lệch -> j chạy tới 4 -> đoạn [2, 3] = ['cd', 'zz'], nhảy i = 4
i = 4: e == e -> khớp, i += 1
i = 5: lệch -> j chạy tới 6 -> đoạn [5, 5] = ['f', 'z'], nhảy i = 6
Kết quả: [['a', 'x'], ['cd', 'zz'], ['f', 'z']]


# 7. Cách 4 — `itertools.groupby`

Bài toán thực chất là: **gom các chỉ số liền nhau có cùng nhãn "lệch / khớp"**, rồi chỉ giữ các nhóm có nhãn "lệch". Đây đúng là việc `groupby` làm.

- Duyệt `range(n)`, nhãn của chỉ số `k` là `s1[k] != s2[k]`.
- Với mỗi nhóm có nhãn `True`, lấy chỉ số đầu `a` và chỉ số cuối `b`, rồi cắt `s1[a:b + 1]`.

In [11]:
from itertools import groupby


def mismatched_groupby(s1: str, s2: str) -> List[List[str]]:
    result = []
    for is_diff, grp in groupby(range(len(s1)), key=lambda k: s1[k] != s2[k]):
        if is_diff:
            idx = list(grp)
            a, b = idx[0], idx[-1]
            result.append([s1[a:b + 1], s2[a:b + 1]])
    return result


print(mismatched_groupby("earth", "empty"))
print(mismatched_groupby("action", "nation"))
print(mismatched_groupby("hot", "hot"))

# Xem groupby gom như thế nào:
for is_diff, grp in groupby(range(5), key=lambda k: "earth"[k] != "empty"[k]):
    print(is_diff, list(grp))

[['ar', 'mp'], ['h', 'y']]
[['ac', 'na']]
[]
False [0]
True [1, 2]
False [3]
True [4]


Cách 4 ngắn và "nói đúng ý" bài toán, nhưng cần biết `groupby`. Trong phỏng vấn, cách 2 hoặc 3 thường được ưa hơn vì thể hiện rõ tư duy.

---

# 8. Phiên bản đúng format LeetCode

Ta chọn cách 3 (hai con trỏ): không có biến trạng thái, không phải nhớ đóng đoạn cuối.

In [12]:
from typing import List


class Solution:
    def findMismatchedSegments(self, s1: str, s2: str) -> List[List[str]]:
        n = len(s1)
        result = []
        i = 0
        while i < n:
            if s1[i] == s2[i]:
                i += 1
                continue
            j = i
            while j < n and s1[j] != s2[j]:
                j += 1
            result.append([s1[i:j], s2[i:j]])
            i = j
        return result

In [13]:
sol = Solution()

tests = [
    ("earth", "empty", [["ar", "mp"], ["h", "y"]], "ví dụ 1"),
    ("action", "nation", [["ac", "na"]], "ví dụ 2"),
    ("hot", "hot", [], "ví dụ 3: giống hệt"),
    ("a", "b", [["a", "b"]], "một ký tự, lệch"),
    ("a", "a", [], "một ký tự, khớp"),
    ("abc", "xyz", [["abc", "xyz"]], "lệch toàn bộ"),
    ("abab", "baba", [["abab", "baba"]], "lệch toàn bộ (xen kẽ)"),
    ("abcdef", "xbzdyf", [["a", "x"], ["c", "z"], ["e", "y"]], "lệch rời rạc"),
    ("abcdef", "abcdeg", [["f", "g"]], "chỉ lệch ở cuối"),
]

for s1, s2, expected, name in tests:
    result = sol.findMismatchedSegments(s1, s2)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:24} -> {result}")

PASS | ví dụ 1                  -> [['ar', 'mp'], ['h', 'y']]
PASS | ví dụ 2                  -> [['ac', 'na']]
PASS | ví dụ 3: giống hệt       -> []
PASS | một ký tự, lệch          -> [['a', 'b']]
PASS | một ký tự, khớp          -> []
PASS | lệch toàn bộ             -> [['abc', 'xyz']]
PASS | lệch toàn bộ (xen kẽ)    -> [['abab', 'baba']]
PASS | lệch rời rạc             -> [['a', 'x'], ['c', 'z'], ['e', 'y']]
PASS | chỉ lệch ở cuối          -> [['f', 'g']]


# 9. Vì sao cách hai con trỏ đúng?

**Bất biến:** ở đầu mỗi vòng `while i < n`, mọi đoạn lệch nằm hoàn toàn trong `[0, i - 1]` đã được ghi vào `result` đúng thứ tự, và vị trí `i - 1` (nếu có) là vị trí **khớp** hoặc là cuối của một đoạn vừa ghi.

- Nếu `s1[i] == s2[i]`: vị trí `i` không thuộc đoạn lệch nào → tăng `i`, bất biến giữ nguyên.
- Nếu `s1[i] != s2[i]`: vì `i - 1` khớp (hoặc `i = 0`), đoạn **không kéo sang trái được** → `i` là đầu của một đoạn lệch tối đa. Vòng `while` trong kéo `j` tới vị trí khớp đầu tiên (hoặc `n`) → đoạn **không kéo sang phải được**. Vậy `[i, j - 1]` đúng là một đoạn lệch tối đa. Sau đó `i = j` là vị trí khớp hoặc `n`, bất biến lại đúng.

Mỗi đoạn được ghi khi `i` đi qua đầu của nó, theo thứ tự trái sang phải → kết quả tự động **sắp theo vị trí bắt đầu**.

---

## Trực giác

> Đoạn lệch tối đa luôn **bắt đầu ngay sau một vị trí khớp** (hoặc ở đầu chuỗi) và **kết thúc ngay trước một vị trí khớp** (hoặc ở cuối chuỗi). Các vị trí khớp chính là "vách ngăn" chia chuỗi thành các đoạn.

# 10. Độ phức tạp

Gọi `n = len(s1)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | mọi `(i, j)`, kiểm tra định nghĩa | `O(n³)` | `O(1)` |
| 2. Biến `start` | máy trạng thái 2 trạng thái | `O(n)` | `O(1)` |
| 3. Hai con trỏ | `i` tìm đầu, `j` kéo hết đoạn | `O(n)` | `O(1)` |
| 4. `groupby` | gom chỉ số theo nhãn lệch/khớp | `O(n)` | `O(n)` trong trường hợp xấu (list chỉ số của nhóm) |

(Không tính bộ nhớ của kết quả: tổng độ dài các chuỗi trả về tối đa `2n` ký tự.)

Lưu ý về "hai vòng lặp lồng nhau mà vẫn `O(n)`": hãy đếm **số lần mỗi vị trí được xét**, không phải đếm số vòng lặp. Ở cách 3, mỗi vị trí được `i` hoặc `j` đi qua đúng một lần.

# 11. Đối chiếu các cách trên dữ liệu ngẫu nhiên

Dùng bảng chữ cái nhỏ (`"ab"`) để có nhiều vị trí khớp lẫn lệch.

In [14]:
import random

random.seed(4106)
for _ in range(3000):
    n = random.randint(1, 12)
    s1 = "".join(random.choice("ab") for _ in range(n))
    s2 = "".join(random.choice("ab") for _ in range(n))
    expected = mismatched_bruteforce(s1, s2)
    assert mismatched_state(s1, s2) == expected, (s1, s2)
    assert mismatched_sentinel(s1, s2) == expected, (s1, s2)
    assert mismatched_two_pointers(s1, s2) == expected, (s1, s2)
    assert mismatched_groupby(s1, s2) == expected, (s1, s2)
    assert sol.findMismatchedSegments(s1, s2) == expected, (s1, s2)

print("3000 bộ test ngẫu nhiên: 6 hàm cho cùng kết quả!")

3000 bộ test ngẫu nhiên: 6 hàm cho cùng kết quả!


# 12. Những lỗi dễ mắc

## Lỗi 1 — Quên đóng đoạn cuối

Ở cách 2, nếu thiếu khối `if start != -1:` sau vòng lặp, đoạn lệch nằm ở **cuối chuỗi** sẽ biến mất. Ví dụ 1 sẽ mất `["h", "y"]`.

In [15]:
def wrong_forget_last(s1: str, s2: str) -> List[List[str]]:
    result = []
    start = -1
    for k in range(len(s1)):
        if s1[k] != s2[k]:
            if start == -1:
                start = k
        elif start != -1:
            result.append([s1[start:k], s2[start:k]])
            start = -1
    return result                        # SAI: quên đoạn còn mở


print("Sai :", wrong_forget_last("earth", "empty"))
print("Đúng:", sol.findMismatchedSegments("earth", "empty"))

Sai : [['ar', 'mp']]
Đúng: [['ar', 'mp'], ['h', 'y']]


## Lỗi 2 — Lệch một đơn vị khi cắt chuỗi

Khi đóng đoạn tại vị trí khớp `k`, viết `s1[start:k + 1]` sẽ **kéo theo cả ký tự khớp** vào đoạn.

In [16]:
def wrong_slice(s1: str, s2: str) -> List[List[str]]:
    result = []
    start = -1
    for k in range(len(s1) + 1):
        diff = k < len(s1) and s1[k] != s2[k]
        if diff and start == -1:
            start = k
        elif not diff and start != -1:
            result.append([s1[start:k + 1], s2[start:k + 1]])   # SAI: thừa 1 ký tự
            start = -1
    return result


print("Sai :", wrong_slice("earth", "empty"))
print("Đúng:", sol.findMismatchedSegments("earth", "empty"))

Sai : [['art', 'mpt'], ['h', 'y']]
Đúng: [['ar', 'mp'], ['h', 'y']]


## Lỗi 3 — Ghi từng vị trí lệch riêng lẻ, không gộp

Code sai:

```python
for k in range(n):
    if s1[k] != s2[k]:
        result.append([s1[k], s2[k]])
```

Ví dụ 1 sẽ ra `[["a","m"], ["r","p"], ["h","y"]]` thay vì `[["ar","mp"], ["h","y"]]`. Đề yêu cầu đoạn **tối đa**.

---

## Lỗi 4 — Hai con trỏ nhưng quên nhảy `i = j`

Nếu sau khi ghi đoạn lại chỉ `i += 1`, con trỏ sẽ dừng ở **giữa** đoạn vừa ghi và ghi thêm các "đoạn con" — ví dụ `"ar"` rồi lại `"r"`. Luôn nhảy qua **cả** đoạn.

---

## Lỗi 5 — Quên điều kiện `j < n` trong vòng `while` trong

`while s1[j] != s2[j]:` sẽ báo `IndexError` khi đoạn lệch kéo tới cuối chuỗi (ví dụ `"ab"` vs `"xy"`). Điều kiện `j < n` phải đứng **trước** để Python dừng kiểm tra sớm (short-circuit).

---

## Lỗi 6 — Trả về tuple hoặc chỉ số thay vì chuỗi

Đề yêu cầu `[[phần của s1, phần của s2], ...]` — list các list hai **chuỗi**. Trả về `[(1, 2), (4, 4)]` hay `[("ar", "mp")]` đều sai định dạng.

# 13. Mẹo nhận diện

Bài thuộc họ **"tách thành các đoạn liên tiếp tối đa"** khi đề có cụm từ:

- *"contiguous"*, *"maximal"*, *"consecutive"*, *"segment"*, *"run"*, *"group of consecutive"*.
- Mỗi vị trí có một **nhãn** (lệch/khớp, chẵn/lẻ, tăng/giảm, ký tự giống nhau...) và đề muốn gom các vị trí liền nhau cùng nhãn.

Ba khuôn viết code đều nên thuộc:

1. **Biến `start` + đóng đoạn** (nhớ đóng đoạn cuối, hoặc dùng lính canh).
2. **Hai con trỏ** `i` tìm đầu, `j` kéo hết đoạn, rồi `i = j`.
3. **`itertools.groupby`** với `key` là nhãn.

Các bài cùng họ: *Summary Ranges* (LeetCode 228), *Max Consecutive Ones* (485), *String Compression* (443), *Count Binary Substrings* (696), *Longest Continuous Increasing Subsequence* (674).

# 14. Biến thể

## Biến thể 1 — Trả về chỉ số `[start, end]` và khoảng cách Hamming

- Khoảng cách Hamming (Hamming distance) = số vị trí lệch.
- Thay vì chuỗi, trả về chỉ số đầu/cuối của từng đoạn.

In [17]:
def mismatch_index_ranges(s1: str, s2: str) -> List[List[int]]:
    result = []
    i, n = 0, len(s1)
    while i < n:
        if s1[i] == s2[i]:
            i += 1
            continue
        j = i
        while j < n and s1[j] != s2[j]:
            j += 1
        result.append([i, j - 1])
        i = j
    return result


def hamming(s1: str, s2: str) -> int:
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))


print(mismatch_index_ranges("earth", "empty"))   # [[1, 2], [4, 4]]
print(hamming("earth", "empty"))                 # 3

[[1, 2], [4, 4]]
3


## Biến thể 2 — Hai chuỗi có thể **khác độ dài**

Giả sử đề nới lỏng: `s1` và `s2` có thể dài khác nhau; vị trí nào chỉ một chuỗi có ký tự thì coi là **lệch**. Ví dụ `s1 = "cat"`, `s2 = "cart"`:

| chỉ số | 0 | 1 | 2 | 3 |
|---|:---:|:---:|:---:|:---:|
| `s1` | c | a | t | (không có) |
| `s2` | c | a | r | t |
| lệch? | | | ✗ | ✗ |

Kết quả: `[["t", "rt"]]`.

Công cụ hữu ích: `itertools.zip_longest(s1, s2, fillvalue=None)` đi song song tới hết chuỗi **dài hơn**, điền `None` cho chuỗi ngắn. Slicing vượt quá độ dài trong Python không báo lỗi mà chỉ cắt tới cuối (`"cat"[2:4] == "t"`), nên phần cắt chuỗi vẫn dùng được nguyên như cũ.

In [18]:
from itertools import zip_longest


def mismatched_any_length(s1: str, s2: str) -> List[List[str]]:
    pairs = list(zip_longest(s1, s2, fillvalue=None))
    n = len(pairs)
    result = []
    i = 0
    while i < n:
        if pairs[i][0] == pairs[i][1]:
            i += 1
            continue
        j = i
        while j < n and pairs[j][0] != pairs[j][1]:
            j += 1
        result.append([s1[i:j], s2[i:j]])      # slicing tự cắt tới cuối chuỗi ngắn
        i = j
    return result


print(list(zip_longest("cat", "cart", fillvalue=None)))
print(mismatched_any_length("cat", "cart"))      # [['t', 'rt']]
print(mismatched_any_length("earth", "empty"))   # giống bài gốc khi cùng độ dài
print(mismatched_any_length("ab", "abcd"))       # [['', 'cd']]

[('c', 'c'), ('a', 'a'), ('t', 'r'), (None, 't')]
[['t', 'rt']]
[['ar', 'mp'], ['h', 'y']]
[['', 'cd']]


# 15. Template tư duy

```python
# Khuôn hai con trỏ: tách các đoạn liên tiếp tối đa thỏa điều kiện good(k)
i, n = 0, len(arr)
while i < n:
    if not good(i):
        i += 1
        continue
    j = i
    while j < n and good(j):
        j += 1
    # đoạn tối đa: [i, j - 1]  -> xử lý ở đây
    i = j
```

```python
# Khuôn groupby
for label, grp in groupby(range(n), key=good):
    if label:
        idx = list(grp)      # idx[0] .. idx[-1] là một đoạn tối đa
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Tìm kết quả cho:

```python
s1 = "planet", s2 = "plates"
```

Gợi ý: lập bảng chỉ số / ký tự / lệch-khớp như ví dụ 1. Kiểm tra bằng cell dưới.

In [19]:
# Kiểm tra Bài 1
trace_state("planet", "plates");

k = 0: p vs p | khớp | start -1 -> -1 | —
k = 1: l vs l | khớp | start -1 -> -1 | —
k = 2: a vs a | khớp | start -1 -> -1 | —
k = 3: n vs t | lệch | start -1 ->  3 | mở đoạn, start = 3
k = 4: e vs e | khớp | start  3 -> -1 | đóng [3, 3] -> ['n', 't']
k = 5: t vs s | lệch | start -1 ->  5 | mở đoạn, start = 5
hết chuỗi: đóng đoạn cuối [5, 5] -> ['t', 's']
Kết quả: [['n', 't'], ['t', 's']]


## Bài 2 — Dự đoán output

```python
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aabbbac")])
print([c1 != c2 for c1, c2 in zip("abc", "abd")])
print("hello"[1:4], "hello"[3:3] == "")
```

In [20]:
print([(k, len(list(g))) for k, g in groupby("aabbbac")])
print([c1 != c2 for c1, c2 in zip("abc", "abd")])
print("hello"[1:4], "hello"[3:3] == "")

[('a', 2), ('b', 3), ('a', 1), ('c', 1)]
[False, False, True]
ell True


## Bài 3 — Hoàn thiện cách biến `start`

In [21]:
def exercise_3_state(s1: str, s2: str) -> List[List[str]]:
    result = []
    start = -1
    for k in range(len(s1)):
        # TODO 1: nếu lệch và chưa mở đoạn -> mở đoạn
        # TODO 2: nếu khớp và đang mở đoạn -> đóng đoạn [start, k - 1]
        pass
    # TODO 3: đóng đoạn cuối nếu còn mở
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_state("earth", "empty") == [["ar", "mp"], ["h", "y"]]
# assert exercise_3_state("hot", "hot") == []

## Bài 4 — Đoạn lệch dài nhất

Trả về độ dài của đoạn lệch dài nhất (`0` nếu hai chuỗi giống nhau).

```python
longest_mismatch("earth", "empty")     -> 2
longest_mismatch("abcdef", "xbzdyf")   -> 1
longest_mismatch("hot", "hot")         -> 0
```

In [22]:
def exercise_4_longest_mismatch(s1: str, s2: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_longest_mismatch("earth", "empty") == 2
# assert exercise_4_longest_mismatch("abcdef", "xbzdyf") == 1
# assert exercise_4_longest_mismatch("hot", "hot") == 0

## Bài 5 — Đoạn **khớp** tối đa

Ngược lại với bài gốc: trả về danh sách các **đoạn khớp** tối đa (chuỗi con chung tại cùng vị trí).

```python
matched_segments("earth", "empty")   -> ["e", "t"]
matched_segments("action", "nation") -> ["tion"]
matched_segments("abc", "xyz")       -> []
```

In [23]:
def exercise_5_matched_segments(s1: str, s2: str) -> List[str]:
    # TODO
    raise NotImplementedError


# assert exercise_5_matched_segments("earth", "empty") == ["e", "t"]
# assert exercise_5_matched_segments("action", "nation") == ["tion"]
# assert exercise_5_matched_segments("abc", "xyz") == []

## Bài 6 — Mini challenge: gộp các đoạn lệch gần nhau

Trả về chỉ số `[start, end]` của các đoạn lệch, nhưng **gộp** hai đoạn lệch liên tiếp nếu khoảng **khớp** giữa chúng dài **không quá `k`** ký tự.

```python
merge_close_mismatches("abcdef", "xbzdyf", 0)  -> [[0, 0], [2, 2], [4, 4]]
merge_close_mismatches("abcdef", "xbzdyf", 1)  -> [[0, 4]]
merge_close_mismatches("aaaaaaa", "baabaab", 1) -> [[0, 0], [3, 3], [6, 6]]
merge_close_mismatches("aaaaaaa", "baabaab", 2) -> [[0, 6]]
```

In [24]:
def exercise_6_merge_close_mismatches(s1: str, s2: str, k: int) -> List[List[int]]:
    # TODO
    raise NotImplementedError


# assert exercise_6_merge_close_mismatches("abcdef", "xbzdyf", 0) == [[0, 0], [2, 2], [4, 4]]
# assert exercise_6_merge_close_mismatches("abcdef", "xbzdyf", 1) == [[0, 4]]
# assert exercise_6_merge_close_mismatches("aaaaaaa", "baabaab", 1) == [[0, 0], [3, 3], [6, 6]]
# assert exercise_6_merge_close_mismatches("aaaaaaa", "baabaab", 2) == [[0, 6]]

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `start = -1` nghĩa là không trong đoạn nào.
- Mức 2: mở đoạn `start = k`; đóng đoạn `result.append([s1[start:k], s2[start:k]])` rồi `start = -1`.
- Mức 3: sau vòng lặp, nếu `start != -1` thì thêm `[s1[start:], s2[start:]]`.

**Bài 4**

- Mức 1: độ dài đoạn `[i, j - 1]` là `j - i`.
- Mức 2: dùng khuôn hai con trỏ, cập nhật `best = max(best, j - i)`.
- Mức 3: hoặc `max((len(a) for a, _ in segments), default=0)`.

**Bài 5**

- Mức 1: chỉ cần đổi điều kiện `!=` thành `==`.
- Mức 2: kết quả là list các chuỗi `s1[i:j]` (vì ở đoạn khớp `s1` và `s2` giống nhau).
- Mức 3: khuôn hai con trỏ với `good(k) = s1[k] == s2[k]`.

**Bài 6**

- Mức 1: trước tiên lấy danh sách đoạn lệch dạng chỉ số (biến thể 1).
- Mức 2: duyệt danh sách; khoảng khớp giữa đoạn trước `[a, b]` và đoạn sau `[c, d]` dài `c - b - 1`.
- Mức 3: nếu `c - b - 1 <= k` thì gộp: sửa `end` của đoạn trước thành `d`; ngược lại thêm đoạn mới.

# 17. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

| k | 0 | 1 | 2 | 3 | 4 | 5 |
|---|:---:|:---:|:---:|:---:|:---:|:---:|
| `s1` | p | l | a | n | e | t |
| `s2` | p | l | a | t | e | s |
| lệch? | | | | ✗ | | ✗ |

Kết quả: `[["n", "t"], ["t", "s"]]`.

## Đáp án Bài 2

```text
[('a', 2), ('b', 3), ('a', 1), ('c', 1)]
[False, False, True]
ell True
```

Để ý `groupby` tạo **hai** nhóm `'a'` vì chúng không liền nhau.

In [25]:
# Đáp án Bài 3
def answer_3_state(s1: str, s2: str) -> List[List[str]]:
    result = []
    start = -1
    for k in range(len(s1)):
        if s1[k] != s2[k]:
            if start == -1:
                start = k
        elif start != -1:
            result.append([s1[start:k], s2[start:k]])
            start = -1
    if start != -1:
        result.append([s1[start:], s2[start:]])
    return result


assert answer_3_state("earth", "empty") == [["ar", "mp"], ["h", "y"]]
assert answer_3_state("hot", "hot") == []
assert answer_3_state("planet", "plates") == [["n", "t"], ["t", "s"]]
print("Bài 3: PASS")

Bài 3: PASS


In [26]:
# Đáp án Bài 4
def answer_4_longest_mismatch(s1: str, s2: str) -> int:
    best = 0
    i, n = 0, len(s1)
    while i < n:
        if s1[i] == s2[i]:
            i += 1
            continue
        j = i
        while j < n and s1[j] != s2[j]:
            j += 1
        best = max(best, j - i)
        i = j
    return best


assert answer_4_longest_mismatch("earth", "empty") == 2
assert answer_4_longest_mismatch("abcdef", "xbzdyf") == 1
assert answer_4_longest_mismatch("hot", "hot") == 0
print("Bài 4: PASS")

Bài 4: PASS


In [27]:
# Đáp án Bài 5
def answer_5_matched_segments(s1: str, s2: str) -> List[str]:
    result = []
    i, n = 0, len(s1)
    while i < n:
        if s1[i] != s2[i]:
            i += 1
            continue
        j = i
        while j < n and s1[j] == s2[j]:
            j += 1
        result.append(s1[i:j])
        i = j
    return result


assert answer_5_matched_segments("earth", "empty") == ["e", "t"]
assert answer_5_matched_segments("action", "nation") == ["tion"]
assert answer_5_matched_segments("abc", "xyz") == []
print("Bài 5: PASS")

Bài 5: PASS


In [28]:
# Đáp án Bài 6
def answer_6_merge_close_mismatches(s1: str, s2: str, k: int) -> List[List[int]]:
    merged = []
    for start, end in mismatch_index_ranges(s1, s2):
        if merged and start - merged[-1][1] - 1 <= k:
            merged[-1][1] = end                  # khoảng khớp đủ ngắn -> gộp
        else:
            merged.append([start, end])
    return merged


assert answer_6_merge_close_mismatches("abcdef", "xbzdyf", 0) == [[0, 0], [2, 2], [4, 4]]
assert answer_6_merge_close_mismatches("abcdef", "xbzdyf", 1) == [[0, 4]]
assert answer_6_merge_close_mismatches("aaaaaaa", "baabaab", 1) == [[0, 0], [3, 3], [6, 6]]
assert answer_6_merge_close_mismatches("aaaaaaa", "baabaab", 2) == [[0, 6]]
print("Bài 6: PASS")

Bài 6: PASS


# 18. Tổng kết

Bài **Find Mismatched Segments** là bài tập chuẩn cho kỹ năng **tách đoạn liên tiếp tối đa**:

- Mỗi vị trí có nhãn **lệch / khớp**.
- Các vị trí khớp là **vách ngăn**; đoạn lệch tối đa nằm giữa hai vách ngăn liên tiếp.
- Viết bằng **biến `start`** (nhớ đóng đoạn cuối / dùng lính canh), **hai con trỏ** (`i = j` sau mỗi đoạn), hoặc **`groupby`**.

## Công thức nhớ nhanh

```text
đoạn chỉ số [i, j - 1]  <->  s[i:j]
i tìm đầu đoạn, j kéo tới vị trí khớp đầu tiên (hoặc n), rồi i = j
```

## Phiên bản nên nhớ

```python
class Solution:
    def findMismatchedSegments(self, s1: str, s2: str) -> List[List[str]]:
        n = len(s1)
        result = []
        i = 0
        while i < n:
            if s1[i] == s2[i]:
                i += 1
                continue
            j = i
            while j < n and s1[j] != s2[j]:
                j += 1
            result.append([s1[i:j], s2[i:j]])
            i = j
        return result
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(1) (không tính kết quả)
```

# 19. Checklist trước khi submit

- [ ] Gộp các vị trí lệch **liền nhau** thành một đoạn (không ghi từng ký tự)?
- [ ] Đoạn lệch ở **cuối chuỗi** có được ghi không? (đóng đoạn cuối / lính canh / hai con trỏ)
- [ ] Slicing đúng: đoạn `[i, j - 1]` là `s[i:j]`, không thừa ký tự khớp?
- [ ] Vòng `while` trong có `j < n` đứng **trước** `s1[j] != s2[j]`?
- [ ] Sau khi ghi đoạn, con trỏ nhảy `i = j` (không `i += 1`)?
- [ ] Định dạng trả về: list các `[chuỗi của s1, chuỗi của s2]`, theo thứ tự bắt đầu tăng dần?
- [ ] Hai chuỗi giống hệt → `[]`?